In [ ]:
# First, downgrade setuptools to a version compatible with existing dependencies (e.g., torch)
# This helps prevent build errors for pandas and xport.
!pip install setuptools==65.5.0 --quiet

# Now, install the xport and pandas libraries
!pip install xport pandas --quiet

import zipfile
import os
import pandas as pd
import xport

# --- Step 1: Unzip the provided file ---
zip_file_path = '/content/Datasets-20260709T003430Z-3-001.zip'
extraction_path = '/content/extracted_datasets'

print(f"Unzipping '{zip_file_path}' to '{extraction_path}'...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)
print("Unzipping complete.")

# --- Step 2: Convert XPT files to CSV ---
converted_csv_files = []

print("\nConverting XPT files to CSV...")
for root, _, files in os.walk(extraction_path):
    for file in files:
        if file.lower().endswith('.xpt'):
            xpt_file_path = os.path.join(root, file)
            csv_file_name = os.path.splitext(file)[0] + '.csv'
            csv_file_path = os.path.join(root, csv_file_name)

            try:
                with open(xpt_file_path, 'rb') as f:
                    df = xport.to_dataframe(f)
                df.to_csv(csv_file_path, index=False)
                converted_csv_files.append(csv_file_path)
                print(f"Converted '{file}' to '{csv_file_name}'")
            except Exception as e:
                print(f"Error converting '{file}': {e}")
        elif file.lower().endswith('.csv'):
            # If there are already CSV files, include them for merging
            csv_file_path = os.path.join(root, file)
            converted_csv_files.append(csv_file_path)
            print(f"Found existing CSV file: '{file}'")

if not converted_csv_files:
    print("No XPT or CSV files found to convert/merge in the extracted directory.")
else:
    # --- Step 3: Merge all CSV datasets into one big one ---
    print("\nMerging all CSV files...")
    all_dataframes = []
    for csv_file in converted_csv_files:
        try:
            df = pd.read_csv(csv_file)
            all_dataframes.append(df)
            print(f"Loaded '{os.path.basename(csv_file)}' for merging.")
        except Exception as e:
            print(f"Error reading CSV file '{os.path.basename(csv_file)}': {e}")

    if all_dataframes:
        merged_df = pd.concat(all_dataframes, ignore_index=True)
        output_merged_csv = '/content/merged_dataset.csv'
        merged_df.to_csv(output_merged_csv, index=False)
        print(f"Successfully merged all datasets into '{output_merged_csv}'")
        print("\nFirst 5 rows of the merged dataset:")
        display(merged_df.head())
        print(f"Total rows in merged dataset: {len(merged_df)}")
    else:
        print("No dataframes were successfully loaded for merging.")